In [62]:
"""v2
╔══════════════════════════════════════════════════════════════════╗
║        VFD Wallets & Bills Payment API — Full Script             ║
║  Account: Dignity Management Concept Limited (1001694651)        ║
╠══════════════════════════════════════════════════════════════════╣
║  1. buy_airtime()          → Bills API  /pay                     ║
║  2. simulate_inflow()      → Wallet API /credit                  ║
║  3. set_transaction_limit()→ Wallet API /transaction/limit       ║
║  4. inter_transfer()       → Wallet API /transfer  (Ecobank)     ║
║  5. intra_transfer()       → Wallet API /transfer  (VFD→VFD)     ║
║  6. transaction_status()   → Wallet API /transactions (TSQ)      ║
║  7. reversal_status()      → Wallet API /transactions/reversal   ║
║                                         (TRSQ)                   ║
╚══════════════════════════════════════════════════════════════════╝
"""

import hashlib
import uuid
import requests

# ─────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────
# ACCESS_TOKEN = (
#     "eyJhbGciOiJIUzUxMiJ9.eyJzdWIiOiI4NzAzMyIsInRva2VuSWQiOiIwZjMzYTM3NS02"
#     "MTVjLTRiMjYtYTA3My1mMDA2OGRjODhmODYiLCJpYXQiOjE3NzQ3MTMzMDUsImV4cCI6"
#     "OTIyMzM3MjAzNjg1NDc3NX0.xrxDUc7Gvp4vBvrjLD2CS7uz2ak7AYAaBTKbhHz7M-GEO"
#     "yWFXchOugw8BsXN9pwhjP32veYWF8_mZjANfU5u6A"
# )

WALLET_BASE = "https://api-devapps.vfdbank.systems/vtech-wallet/api/v2/wallet2"
BILLS_BASE  = "https://api-devapps.vfdbank.systems/vtech-bills/api/v2/billspaymentstore"
WALLET_NAME = "Dignity"   # used as reference prefix

# Sender / FROM account  (Dignity Management Concept Limited)
FROM_ACCOUNT    = "1001694651"
FROM_CLIENT_ID  = "154658"
FROM_CLIENT     = "Dignity Management Concept Limited"
FROM_SAVINGS_ID = "169465"
FROM_BVN        = ""

HEADERS = {"AccessToken": ACCESS_TOKEN}


# ─────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────
def _ref() -> str:
    """Generate a unique wallet-prefixed transaction reference."""
    return f"{WALLET_NAME}-{uuid.uuid4().hex[:16].upper()}"


def _sha512(from_acct: str, to_acct: str) -> str:
    """Generate the required SHA-512 transfer signature."""
    return hashlib.sha512(f"{from_acct}{to_acct}".encode()).hexdigest()


def _banner(title: str):
    print("\n" + "=" * 62)
    print(f"  {title}")
    print("=" * 62)


# ═════════════════════════════════════════════════════════════════
#  1a. BUY AIRTIME
#     Bills Payment API  →  POST /pay
#     Flow: billerlist → billerItems → pay
# ═════════════════════════════════════════════════════════════════
def buy_airtimev2(phone_number: str, amount: int, network: str = "MTN"):
    """
    Purchase airtime for any Nigerian network.

    Args:
        phone_number : e.g. "08012345678"
        amount       : airtime value in Naira e.g. 500
        network      : "MTN" | "AIRTEL" | "GLO" | "9MOBILE"
    """
    _banner("SCRIPT 1 — BUY AIRTIME")

    # Step 1 — Biller list
    print(f"\n[ Step 1 ] Fetching Airtime billers...")
    r = requests.get(f"{BILLS_BASE}/billerlist", headers=HEADERS,
                     params={"categoryName": "Airtime"})
    r.raise_for_status()
    billers = r.json().get("data", [])
    print(f"           Available: {[b['name'] for b in billers]}")

    biller = next((b for b in billers if network.upper() in b["name"].upper()), None)
    if not biller:
        raise ValueError(f"Network '{network}' not found. Available: {[b['name'] for b in billers]}")

    biller_id, division_id, product_id = biller["id"], biller["division"], biller["product"]
    print(f"           Selected : {biller['name']} | id={biller_id} | div={division_id} | prod={product_id}")

    # Step 2 — Biller items
    print(f"\n[ Step 2 ] Fetching biller items...")
    r2 = requests.get(f"{BILLS_BASE}/billerItems", headers=HEADERS,
                      params={"billerId": biller_id, "divisionId": division_id, "productId": product_id})
    r2.raise_for_status()
    items = r2.json().get("data", {}).get("paymentitems", [])
    if not items:
        raise ValueError("No payment items returned.")
    payment_code = items[0].get("paymentCode", biller_id)
    print(f"           Item     : {items[0].get('paymentitemname')} | paymentCode={payment_code}")

    # Step 3 — Pay
    ref     = _ref()
    payload = {
        "customerId":  phone_number,
        "amount":      str(amount),
        "division":    division_id,
        "paymentItem": payment_code,
        "productId":   product_id,
        "billerId":    biller_id,
        "reference":   ref,
        "phoneNumber": phone_number,
    }
    print(f"\n[ Step 3 ] Paying ₦{amount:,} airtime → {phone_number} ({network})")
    print(f"           Reference : {ref}")

    r3 = requests.post(f"{BILLS_BASE}/pay", headers=HEADERS, json=payload)
    r3.raise_for_status()
    res = r3.json()

    if res.get("status") == "00":
        print(f"\n✅ Airtime Successful! Reference: {res['data'].get('reference')}")
    else:
        print(f"\n❌ Failed [{res.get('status')}]: {res.get('message')}")
    return res

# ═════════════════════════════════════════════════════════════
# SCRIPT 1 — BUY AIRTIME
# API: POST {BILLS_BASE_URL}/pay
# Flow: billercategory → billerlist → billerItems → pay
# ═════════════════════════════════════════════════════════════
def buy_airtime(phone_number: str, amount: int, network: str = "MTN"):
    """
    Buy airtime for a phone number.

    Args:
        phone_number : recipient phone number e.g. "08012345678"
        amount       : airtime amount in Naira e.g. 500
        network      : "MTN", "AIRTEL", "GLO", "9MOBILE"
    """
    print("\n" + "="*60)
    print("SCRIPT 1 — BUY AIRTIME")
    print("="*60)

    # ── Step 1: Get biller list for Airtime ──────────────────
    print("\n[ Step 1 ] Fetching Airtime biller list...")
    resp = requests.get(
        f"{BILLS_BASE_URL}/billerlist",
        headers=HEADERS,
        params={"categoryName": "Airtime"}
    )
    resp.raise_for_status()
    biller_data = resp.json()
    print(f"           Status: {biller_data.get('status')} — {biller_data.get('message')}")

    # Pick the biller matching the requested network
    billers = biller_data.get("data", [])
    biller = next(
        (b for b in billers if network.upper() in b.get("name", "").upper()),
        None
    )
    if not biller:
        # Fallback: print available billers and raise
        print(f"           Available billers: {[b['name'] for b in billers]}")
        raise Exception(f"Network '{network}' not found in biller list.")

    biller_id   = biller["id"]
    division_id = biller["division"]
    product_id  = biller["product"]
    print(f"           Biller: {biller['name']} | id={biller_id} | division={division_id} | product={product_id}")

    # ── Step 2: Get biller items ─────────────────────────────
    print("\n[ Step 2 ] Fetching biller items...")
    resp2 = requests.get(
        f"{BILLS_BASE_URL}/billerItems",
        headers=HEADERS,
        params={
            "billerId":   biller_id,
            "divisionId": division_id,
            "productId":  product_id
        }
    )
    resp2.raise_for_status()
    items_data = resp2.json()
    payment_items = items_data.get("data", {}).get("paymentitems", [])
    if not payment_items:
        raise Exception("No payment items returned.")

    item = payment_items[0]  # Use first item (airtime has one flexible item)
    payment_code = item.get("paymentCode", biller_id)
    print(f"           Item: {item.get('paymentitemname')} | paymentCode={payment_code}")

    # ── Step 3: Pay ──────────────────────────────────────────
    unique_ref = f"{WALLET_NAME}-{uuid.uuid4().hex[:12].upper()}"
    payload = {
        "customerId":   phone_number,
        "amount":       str(amount),
        "division":     division_id,
        "paymentItem":  payment_code,
        "productId":    product_id,
        "billerId":     biller_id,
        "reference":    unique_ref,
        "phoneNumber":  phone_number
    }

    print(f"\n[ Step 3 ] Paying ₦{amount:,} airtime to {phone_number} ({network})")
    print(f"           Reference: {unique_ref}")
    resp3 = requests.post(f"{BILLS_BASE_URL}/pay", headers=HEADERS, json=payload)
    resp3.raise_for_status()
    result = resp3.json()

    if result.get("status") == "00":
        print(f"\n✅ Airtime Purchase Successful!")
        print(f"   Reference: {result['data'].get('reference')}")
    else:
        print(f"\n❌ Failed: [{result.get('status')}] {result.get('message')}")

    return result


# ═════════════════════════════════════════════════════════════════
#  2. SIMULATE INFLOW  (DEV only)
#     Wallet API  →  POST /credit
# ═════════════════════════════════════════════════════════════════
def simulate_inflow(account_no: str, amount: int,
                    narration: str = "Test inflow credit"):
    """
    Simulate an inward bank credit to a wallet account (test env only).

    Args:
        account_no : VFD wallet account to receive funds e.g. "1001694651"
        amount     : amount in Naira
        narration  : sender narration
    """
    _banner("SCRIPT 2 — SIMULATE INFLOW  (/credit)")

    payload = {
        "amount":          str(amount),
        "accountNo":       account_no,
        "senderAccountNo": "5050104070",  # doc-specified test sender
        "senderBank":      "000002",       # Keystone Bank — doc-specified
        "senderNarration": narration,
    }
    print(f"\n  Crediting    : ₦{amount:,} → Account {account_no}")
    print(f"  Sender Acct  : 5050104070  |  Sender Bank: 000002 (Keystone)")
    print(f"  Narration    : {narration}")

    r = requests.post(f"{WALLET_BASE}/credit", headers=HEADERS, json=payload)
    r.raise_for_status()
    res = r.json()

    if res.get("status") == "00":
        print(f"\n✅ Inflow Simulated! Message: {res.get('message')}")
    else:
        print(f"\n❌ Failed [{res.get('status')}]: {res.get('message')}")
    print(f"  Full response: {res}")
    return res


# ═════════════════════════════════════════════════════════════════
#  3. SET TRANSACTION LIMIT
#     Wallet API  →  POST /transaction/limit
# ═════════════════════════════════════════════════════════════════
def set_transaction_limit(account_number: str,
                          transaction_limit: int = 500_000,
                          daily_limit: int = 500_000):
    """
    Update the per-transaction and daily withdrawal limits for an account.

    Args:
        account_number    : e.g. "1001694651"
        transaction_limit : max single-transaction amount in Naira
        daily_limit       : max cumulative daily withdrawal in Naira
    """
    _banner("SCRIPT 3 — SET TRANSACTION LIMIT  (/transaction/limit)")

    payload = {
        "accountNumber":    account_number,
        "transactionLimit": str(transaction_limit),
        "dailyLimit":       str(daily_limit),
    }
    print(f"\n  Account           : {account_number}")
    print(f"  Transaction Limit : ₦{transaction_limit:,}")
    print(f"  Daily Limit       : ₦{daily_limit:,}")

    r = requests.post(f"{WALLET_BASE}/transaction/limit", headers=HEADERS, json=payload)
    r.raise_for_status()
    res = r.json()

    if res.get("status") == "00":
        print(f"\n✅ Limits Updated! Message: {res.get('message')}")
    else:
        print(f"\n❌ Failed [{res.get('status')}]: {res.get('message')}")
    print(f"  Full response: {res}")
    return res


# ═════════════════════════════════════════════════════════════════
#  4. INTER-BANK TRANSFER  (VFD → Other Bank)
#     Wallet API  →  POST /transfer   transferType="inter"
# ═════════════════════════════════════════════════════════════════
def inter_transfer(to_account: str, bank_code: str, amount: int,
                   remark: str = "Inter-bank transfer"):
    """
    Transfer funds from the Dignity VFD account to any external bank.

    Args:
        to_account : beneficiary account number e.g. "2112279111"
        bank_code  : beneficiary bank NIP code  e.g. "000010" (Ecobank)
        amount     : amount in Naira
        remark     : transaction narration
    """
    _banner("SCRIPT 4 — INTER-BANK TRANSFER  (transferType=inter)")

    # Recipient lookup
    print(f"\n[ Step 1 ] Recipient lookup: {to_account} @ bank {bank_code}...")
    r = requests.get(f"{WALLET_BASE}/transfer/recipient", headers=HEADERS,
                     params={"accountNo": to_account, "bank": bank_code,
                             "transfer_type": "inter"})
    r.raise_for_status()
    td = r.json()
    print(f"           Response: {td}")
    if td.get("status") != "00":
        raise Exception(f"Recipient lookup failed [{td['status']}]: {td['message']}")

    info       = td["data"]
    to_session = info["account"].get("id")   # mandatory for inter
    print(f"           TO: {info.get('name')} | clientId={info.get('clientId')} | id={to_session}")

    # Signature
    sig = _sha512(FROM_ACCOUNT, to_account)
    print(f"\n[ Step 2 ] Signature (SHA512): {sig[:40]}...")

    # Transfer
    ref     = _ref()
    payload = {
        "fromAccount":           FROM_ACCOUNT,
        "uniqueSenderAccountId": "",
        "fromClientId":          FROM_CLIENT_ID,
        "fromClient":            FROM_CLIENT,
        "fromSavingsId":         FROM_SAVINGS_ID,
        "fromBvn":               FROM_BVN,
        "toClientId":            info.get("clientId", ""),
        "toClient":              info.get("name"),
        "toSavingsId":           "",          # not mandatory for inter
        "toSession":             to_session,  # mandatory for inter
        "toBvn":                 info.get("bvn", ""),
        "toAccount":             to_account,
        "toBank":                bank_code,
        "signature":             sig,
        "amount":                str(amount),
        "remark":                remark,
        "transferType":          "inter",
        "reference":             ref,
    }
    print(f"\n[ Step 3 ] Sending ₦{amount:,} → {to_account} ({info.get('name')}) | Ref: {ref}")

    r2 = requests.post(f"{WALLET_BASE}/transfer", headers=HEADERS, json=payload)
    r2.raise_for_status()
    res = r2.json()

    if res.get("status") == "00":
        print(f"\n✅ Transfer Successful!")
        print(f"   txnId     : {res['data'].get('txnId')}")
        print(f"   sessionId : {res['data'].get('sessionId', 'N/A')}")
        print(f"   reference : {res['data'].get('reference', ref)}")
    else:
        print(f"\n❌ Failed [{res.get('status')}]: {res.get('message')}")
        print(f"   Full response: {res}")
    return res


# ═════════════════════════════════════════════════════════════════
#  5. INTRA-BANK TRANSFER  (VFD → VFD)
#     Wallet API  →  POST /transfer   transferType="intra"
# ═════════════════════════════════════════════════════════════════
def intra_transfer(to_account: str = "1001696569", amount: int = 30_000,
                   remark: str = "Intra transfer"):
    """
    Transfer funds between two VFD wallet accounts.

    Args:
        to_account : VFD beneficiary account number
        amount     : amount in Naira
        remark     : transaction narration
    """
    _banner("SCRIPT 5 — INTRA-BANK TRANSFER  (transferType=intra)")

    # Recipient lookup
    print(f"\n[ Step 1 ] Recipient lookup: {to_account} @ VFD (999999)...")
    r = requests.get(f"{WALLET_BASE}/transfer/recipient", headers=HEADERS,
                     params={"accountNo": to_account, "bank": "999999",
                             "transfer_type": "intra"})
    r.raise_for_status()
    td = r.json()
    print(f"           Response: {td}")
    if td.get("status") != "00":
        raise Exception(f"Recipient lookup failed [{td['status']}]: {td['message']}")

    info         = td["data"]
    to_savings_id = info["account"].get("id")   # mandatory for intra
    print(f"           TO: {info.get('name')} | clientId={info.get('clientId')} | savingsId={to_savings_id}")

    # Signature
    sig = _sha512(FROM_ACCOUNT, to_account)
    print(f"\n[ Step 2 ] Signature (SHA512): {sig[:40]}...")

    # Transfer
    ref     = _ref()
    payload = {
        "fromAccount":           FROM_ACCOUNT,
        "uniqueSenderAccountId": "",
        "fromClientId":          FROM_CLIENT_ID,
        "fromClient":            FROM_CLIENT,
        "fromSavingsId":         FROM_SAVINGS_ID,
        "fromBvn":               FROM_BVN,
        "toClientId":            info.get("clientId"),
        "toClient":              info.get("name"),
        "toSavingsId":           to_savings_id,  # mandatory for intra
        "toSession":             "",
        "toBvn":                 info.get("bvn", ""),
        "toAccount":             to_account,
        "toBank":                "999999",        # VFD bank code
        "signature":             sig,
        "amount":                str(amount),
        "remark":                remark,
        "transferType":          "intra",
        "reference":             ref,
    }
    print(f"\n[ Step 3 ] Sending ₦{amount:,} → {to_account} ({info.get('name')}) | Ref: {ref}")

    r2 = requests.post(f"{WALLET_BASE}/transfer", headers=HEADERS, json=payload)
    r2.raise_for_status()
    res = r2.json()

    if res.get("status") == "00":
        print(f"\n✅ Transfer Successful!")
        print(f"   txnId : {res['data'].get('txnId')}")
    else:
        print(f"\n❌ Failed [{res.get('status')}]: {res.get('message')}")
        print(f"   Full response: {res}")
    return res


# ═════════════════════════════════════════════════════════════════
#  6. TRANSACTION STATUS QUERY  (TSQ)
#     Wallet API  →  GET /transactions?reference={reference}
#                        /transactions?sessionId={sessionId}
#
#  Use this to check whether a transfer succeeded, is pending,
#  or failed — especially for status codes 01 / 02 (pending).
# ═════════════════════════════════════════════════════════════════
def transaction_status(reference: str = None, session_id: str = None):
    """
    Query the status of a transaction by reference or sessionId.

    Pass one of:
        reference  : the unique reference used during the transfer
                     e.g. "Dignity-ABCD1234EFGH5678"
        session_id : the NIP sessionId returned on inter-bank transfers
                     e.g. "090110220924210740278805531934"

    Response fields:
        TxnId             — internal transaction ID
        amount            — transaction amount
        accountNo         — beneficiary account
        fromAccountNo     — sender account
        transactionStatus — "00" success | "99" failed | "01"/"02" pending
        transactionDate   — timestamp
        toBank / fromBank — bank codes
        sessionId         — NIP session ID (inter-bank only)
    """
    _banner("SCRIPT 6 — TRANSACTION STATUS QUERY  (TSQ)")

    if not reference and not session_id:
        raise ValueError("Provide at least one of: reference or session_id")

    params = {}
    if reference:
        params["reference"] = reference
        print(f"\n  Querying by Reference : {reference}")
    else:
        params["sessionId"] = session_id
        print(f"\n  Querying by SessionId : {session_id}")

    r = requests.get(f"{WALLET_BASE}/transactions", headers=HEADERS, params=params)
    r.raise_for_status()
    res = r.json()

    if res.get("status") == "00":
        d = res.get("data", {})
        print(f"\n✅ Transaction Found!")
        print(f"   TxnId             : {d.get('TxnId')}")
        print(f"   Amount            : ₦{float(d.get('amount', 0)):,.2f}")
        print(f"   From Account      : {d.get('fromAccountNo')}")
        print(f"   To Account        : {d.get('accountNo')}")
        print(f"   Transaction Status: {d.get('transactionStatus')}  "
              f"({'✅ SUCCESS' if d.get('transactionStatus') == '00' else '❌ FAILED' if d.get('transactionStatus') == '99' else '⏳ PENDING'})")
        print(f"   Transaction Date  : {d.get('transactionDate')}")
        print(f"   From Bank         : {d.get('fromBank')}")
        print(f"   To Bank           : {d.get('toBank')}")
        print(f"   Session ID        : {d.get('sessionId', 'N/A')}")
    else:
        print(f"\n❌ Query Failed [{res.get('status')}]: {res.get('message')}")

    print(f"\n  Full response: {res}")
    return res


# ═════════════════════════════════════════════════════════════════
#  7. TRANSACTION REVERSAL STATUS QUERY  (TRSQ)
#     Wallet API  →  GET /transactions/reversal?reference={reference}
#
#  Use this after a failed/disputed transfer to check whether
#  the funds were reversed back to the sender.
# ═════════════════════════════════════════════════════════════════
def reversal_status(reference: str):
    """
    Query the reversal status of a transaction.

    Args:
        reference : the unique transaction reference (mandatory)
                    e.g. "Dignity-ABCD1234EFGH5678"

    Response fields:
        amount         — amount reversed
        reversalStatus — "00" reversed successfully
        reversalDate   — date reversal was completed
        reversalId     — reversal record ID
        sessionId      — original NIP session ID
    """
    _banner("SCRIPT 7 — TRANSACTION REVERSAL STATUS QUERY  (TRSQ)")

    print(f"\n  Reference : {reference}")

    r = requests.get(f"{WALLET_BASE}/transactions/reversal",
                     headers=HEADERS,
                     params={"reference": reference})
    r.raise_for_status()
    res = r.json()

    if res.get("status") == "00":
        d = res.get("data", {})
        status_label = "✅ REVERSED" if d.get("reversalStatus") == "00" else "⏳ PENDING / NOT YET REVERSED"
        print(f"\n✅ Reversal Record Found!")
        print(f"   Reversal Status : {d.get('reversalStatus')}  ({status_label})")
        print(f"   Amount Reversed : ₦{float(d.get('amount', 0)):,.2f}")
        print(f"   Reversal Date   : {d.get('reversalDate', 'N/A')}")
        print(f"   Reversal ID     : {d.get('reversalId', 'N/A')}")
        print(f"   Session ID      : {d.get('sessionId', 'N/A')}")
    else:
        print(f"\n❌ Query Failed [{res.get('status')}]: {res.get('message')}")

    print(f"\n  Full response: {res}")
    return res
# ═════════════════════════════════════════════════════════════════
#  8. Baalance Inquiry
#     Wallet API  →  GET /account/enquiry?accountNumber={account_number}
#
#  Use this to check the balance of an account, especially after transfers or reversals to confirm whether
#  the expected amount was credited/debited.
# ═════════════════════════════════════════════════════════════════

def balance_inquiry(account_number: str):
    """
    Inquire about the balance of an account.
    """
    print("\n" + "="*60)
    print("SCRIPT 5 — BALANCE INQUIRY")
    print("="*60)

    print(f"\n[ Step 1 ] Inquiring about balance for account: {account_number}")

    try:
        # Use GET as per API docs
        resp = requests.get(
            f"{WALLET_BASE_URL}/account/enquiry?accountNumber={account_number}",
            headers=HEADERS,
            timeout=30
        )
        resp.raise_for_status()
        result = resp.json()

        if result.get("status") == "00":
            print(f"\n✅ Balance Inquired Successfully!")
            print(f"   Message: {result.get('message')}")
            print(f"   Account No: {result['data']['accountNo']}")
            print(f"   Balance: {result['data']['accountBalance']}")
            print(f"   Client: {result['data']['client']}")
        else:
            print(f"\n❌ Failed: [{result.get('status')}] {result.get('message')}")

        print(f"   Full Response: {result}")
        return result

    except Exception as e:
        print(f"\n❌ Request failed: {e}")
        return {"status": "error", "message": str(e)}


# ═════════════════════════════════════════════════════════════════
#  MAIN — Run each script individually or together
# ═════════════════════════════════════════════════════════════════
if __name__ == "__main__":

    # ── 1. Buy Airtime ────────────────────────────────────────
    buy_airtime(
        phone_number="08012345678",
        amount=500,
        network="AIRTEL",   # MTN | AIRTEL | GLO | 9MOBILE
    )

    # ── 2. Simulate Inflow (DEV only) ─────────────────────────
    simulate_inflow(
        account_no=FROM_ACCOUNT,
        amount=8_000,
        narration="Test inflow from Keystone Bank",
    )

    # # ── 3. Set Transaction Limit ──────────────────────────────
    # set_transaction_limit(
    #     account_number=FROM_ACCOUNT,
    #     transaction_limit=500_000,
    #     daily_limit=500_000,
    # )

    # ── 4. Inter-Bank Transfer → Ecobank 2112279111 ───────────
    # inter_transfer(
    #     to_account="2112279111",
    #     bank_code="000010",   # Ecobank Nigeria
    #     amount=30_000,
    #     remark="Payment to Ecobank",
    # )

    # ── 5. Intra Transfer → VFD account 1001696569 ────────────
    intra_transfer(
        to_account="1001696569",
        amount=10_000,
        remark="Intra VFD transfer",
    )

    # ── 6. TSQ — check status of a past transfer ──────────────
    # Replace with a real reference or sessionId from a transfer above
    transaction_status(reference="Dignity-ABCD1234EFGH5678")
    # OR query by sessionId:
    # transaction_status(session_id="090110220924210740278805531934")

    # ── 7. TRSQ — check reversal status ───────────────────────
    # Replace with the reference of the failed/disputed transaction
    reversal_status(reference="Dignity-ABCD1234EFGH5678")
    # ── 8. Balance Inquiry ─────────────────────────────────────
    balance_inquiry("1001696569")


SCRIPT 1 — BUY AIRTIME

[ Step 1 ] Fetching Airtime biller list...
           Status: 00 — Successfully Returned Biller list
           Biller: AIRTEL | id=airng | division=C | product=423

[ Step 2 ] Fetching biller items...
           Item: AIRNG | paymentCode=airng

[ Step 3 ] Paying ₦500 airtime to 08012345678 (AIRTEL)
           Reference: Dignity-1FF660E4FCCC

✅ Airtime Purchase Successful!
   Reference: Dignity-1FF660E4FCCC

  SCRIPT 2 — SIMULATE INFLOW  (/credit)

  Crediting    : ₦8,000 → Account 1001694651
  Sender Acct  : 5050104070  |  Sender Bank: 000002 (Keystone)
  Narration    : Test inflow from Keystone Bank

✅ Inflow Simulated! Message: Successfully notified
  Full response: {'status': '00', 'message': 'Successfully notified'}

  SCRIPT 5 — INTRA-BANK TRANSFER  (transferType=intra)

[ Step 1 ] Recipient lookup: 1001696569 @ VFD (999999)...
           Response: {'status': '00', 'message': 'Account Found', 'data': {'name': 'Dignity Management Concept Limited-Victory Bu

HTTPError: 404 Client Error: Not Found for url: https://api-devapps.vfdbank.systems/vtech-wallet/api/v2/wallet2/transactions?reference=Dignity-ABCD1234EFGH5678